# Capstone II: Autonomous Report Agent

## CareerAlign GenAI Course - Module 16

Build an autonomous multi-agent system that researches a topic, gathers data from multiple sources, synthesizes findings, and produces a structured report -- all with minimal human intervention.

---

### Learning Objectives

By the end of this notebook, you will be able to:

1. **Design a multi-agent architecture** using the supervisor pattern with specialized agents (Planner, Researcher, Synthesizer, Writer, Reviewer)
2. **Build research tools** for web search, RAG retrieval, and data API access
3. **Orchestrate agents with LangGraph** using state machines, conditional edges, and checkpointing
4. **Generate structured reports** with Pydantic models and template-based formatting
5. **Implement guardrails** for budget tracking, content filtering, and citation validation
6. **Add human-in-the-loop** approval gates at critical pipeline stages
7. **Evaluate report quality** using automated metrics and LLM-as-judge scoring
8. **Export reports** to Markdown and HTML formats

### Prerequisites

- Modules 09 (Agents), 03 (LLM APIs), 06 (Prompt Engineering), 07 (RAG), 10 (Evaluation)
- Familiarity with LangGraph state machines and tool use patterns

### Key Technologies

| Component | Technology |
|-----------|------------|
| Orchestration | LangGraph (StateGraph, conditional edges, checkpointing) |
| Agent Brains | OpenAI GPT-4o via `openai` SDK |
| Tool Framework | `langchain-core` tools |
| Structured Output | Pydantic models |
| Report Formats | Markdown, HTML |

---

## 1. Environment Setup & Dependencies

In [ ]:
# Install required packages
# !pip install langgraph langchain-core openai pydantic httpx markdown

In [ ]:
import os
import json
import re
from datetime import datetime
from typing import TypedDict, Annotated
from dataclasses import dataclass
from pathlib import Path

from openai import AsyncOpenAI
from pydantic import BaseModel, Field
import httpx

from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, add_messages
from langgraph.checkpoint.memory import MemorySaver

# ── API Key Configuration ──
# Set your OpenAI API key as an environment variable before running.
# Example: export OPENAI_API_KEY="sk-..."
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "your-api-key-here")

print("All imports successful.")
print(f"Notebook started at: {datetime.now().isoformat()}")

---

## 2. Project Architecture Overview

The Autonomous Report Agent uses a **supervisor pattern** where a central LangGraph state machine orchestrates five specialized agents:

```
START --> [Planner] --> [Human Review] --> [Researcher] --> [Synthesizer] --> [Writer] --> [Reviewer]
                                                                ^                            |
                                                                |         (revise)           |
                                                                +----------------------------+
                                                                              |
                                                                          (pass) --> END
```

**Agent Roles:**

| Agent | Role | Tools |
|-------|------|-------|
| **Planner** | Breaks topic into research questions and sections | None (LLM reasoning only) |
| **Researcher** | Gathers information using ReAct loop | Web Search, RAG Retrieval, Data APIs |
| **Synthesizer** | Analyzes findings, identifies themes and gaps | None (LLM reasoning only) |
| **Writer** | Produces structured Markdown report | None (LLM generation) |
| **Reviewer** | Quality checks: citations, accuracy, completeness | None (LLM evaluation) |

**Design Decision:** We use the supervisor pattern rather than a fully autonomous agent swarm because it provides predictability and debuggability. Each agent has a clear role, the workflow is deterministic, and you can inspect the state at any point to understand what happened.

---

## 3. State Definition

The state is the central data structure that flows through every node. It uses a `TypedDict` to ensure type safety and accumulates data from each stage of the pipeline.

In [ ]:
class ReportState(TypedDict):
    """Central state for the report generation pipeline.

    Each agent reads from and writes to specific fields:
    - Planner writes: plan
    - Researcher writes: research_notes
    - Synthesizer writes: analysis
    - Writer writes: report
    - Reviewer writes: review_passed, review_feedback
    """
    topic: str                                      # User's report topic
    plan: dict                                      # Research plan from Planner
    plan_approved: bool                             # Human approval flag
    research_notes: list[dict]                      # Raw findings from Researcher
    analysis: str                                   # Synthesized analysis
    report: str                                     # Final report (Markdown)
    review_feedback: str                            # Reviewer's feedback
    review_passed: bool                             # Quality gate
    revision_count: int                             # Track revision loops
    messages: Annotated[list, add_messages]         # Agent messages
    budget_used: float                              # Cost tracking ($)
    errors: list[str]                               # Error log


# Show the state fields
print("ReportState fields:")
for field_name, field_type in ReportState.__annotations__.items():
    print(f"  {field_name}: {field_type}")

The state grows as each agent contributes its output. Key design choices:

- **`revision_count`** prevents infinite loops -- after 3 revisions, the system accepts the report as-is and flags it for human review.
- **`budget_used`** tracks accumulated API costs to enforce spending limits.
- **`messages`** uses LangGraph's `add_messages` reducer to append rather than overwrite.
- **`errors`** captures any issues for debugging.

---

## 4. Data Collection Agent -- Research Tools

The Research agent is the workhorse of the system. It uses the **ReAct pattern** (Reason, Act, Observe) to gather information through a set of well-defined tools.

Each tool is a decorated function with a clear docstring that the LLM reads to understand when and how to use it.

In [ ]:
# ── Research Tools ──
# These tools give the Research agent access to external data sources.

TAVILY_KEY = os.environ.get("TAVILY_API_KEY", "your-tavily-key")


@tool
async def web_search(query: str) -> str:
    """Search the web for information on a topic.
    Returns summaries of the top 5 results."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://api.tavily.com/search",
            params={"query": query, "max_results": 5, "api_key": TAVILY_KEY},
        )
        results = resp.json()["results"]
    return "\n\n".join(
        f"[{r['title']}]({r['url']})\n{r['content']}" for r in results
    )


@tool
async def rag_retrieve(query: str, collection: str = "default") -> str:
    """Search uploaded documents using RAG.
    Returns relevant passages with citations."""
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            "http://localhost:8000/query",
            json={"question": query, "collection": collection},
        )
        data = resp.json()
    sources = "\n".join(
        f"- {s['filename']} p.{s['page']}" for s in data.get("sources", [])
    )
    return f"{data['answer']}\n\nSources:\n{sources}"


@tool
async def fetch_data(url: str, description: str) -> str:
    """Fetch structured data from a public API endpoint.
    Provide the URL and a description of what you expect."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(url, timeout=15.0)
        resp.raise_for_status()
    # Truncate large responses to prevent context overflow
    text = resp.text[:5000]
    return f"Data from {url}:\n{text}"


@tool
def save_note(title: str, content: str, sources: list[str]) -> str:
    """Save a research finding as a structured note."""
    return json.dumps({"title": title, "content": content, "sources": sources})


RESEARCH_TOOLS = [web_search, rag_retrieve, fetch_data, save_note]

print("Research tools defined:")
for t in RESEARCH_TOOLS:
    print(f"  - {t.name}: {t.description[:60]}...")

### Research Agent Class

The `ResearchAgent` wraps the tools in a ReAct loop with budget controls. It loops through a maximum number of steps, calling tools as needed. When it calls `save_note`, the structured finding is captured and returned.

The agent autonomously decides which tools to call based on the question -- it might search the web first, find a relevant API endpoint, fetch data from it, and then save a synthesized note.

In [ ]:
class ResearchAgent:
    """A ReAct-based research agent that uses tools to gather information.

    The agent reasons about what information it needs, selects a tool,
    observes the results, and decides whether to continue or stop.
    """

    def __init__(self, tools, max_steps: int = 10):
        self.client = AsyncOpenAI()
        self.tools = {t.name: t for t in tools}
        self.max_steps = max_steps
        self.tool_schemas = [
            {
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.args_schema.schema(),
                },
            }
            for t in tools
        ]

    async def research_question(
        self, question: str, context: str = ""
    ) -> list[dict]:
        """Research a single question using the ReAct pattern.

        Args:
            question: The research question to investigate.
            context: Additional context (e.g., report title, section).

        Returns:
            A list of structured research notes.
        """
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a thorough research agent. For the "
                    "given question, use the available tools to gather "
                    "comprehensive information. Save each distinct "
                    "finding as a note with proper source citations. "
                    "Stop when you have sufficient evidence to answer "
                    "the question thoroughly."
                ),
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {question}",
            },
        ]

        notes = []
        for step in range(self.max_steps):
            resp = await self.client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                tools=self.tool_schemas,
                tool_choice="auto",
            )
            msg = resp.choices[0].message
            messages.append(msg)

            if not msg.tool_calls:
                break  # Agent decided it has enough info

            for tc in msg.tool_calls:
                tool_fn = self.tools[tc.function.name]
                args = json.loads(tc.function.arguments)
                result = await tool_fn.ainvoke(args)

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": str(result),
                    }
                )

                # Capture saved notes
                if tc.function.name == "save_note":
                    notes.append(json.loads(result))

        return notes


print("ResearchAgent class defined.")
print(f"  Max steps per question: 10")
print(f"  Available tools: {list(RESEARCH_TOOLS[0].name for _ in [0])}...")  # noqa
print(f"  Tools: {[t.name for t in RESEARCH_TOOLS]}")

---

## 5. Analysis Agent -- Data Processing & Insights

The Synthesizer agent takes raw research notes and produces structured analysis. It identifies key themes, contradictions, and gaps across all the findings.

In [ ]:
# ── Node Functions (Agent Implementations) ──


async def plan_node(state: ReportState) -> dict:
    """Planner agent: Break topic into research questions.

    Outputs a structured plan with sections, questions per section,
    and an estimate of sources needed.
    """
    client = AsyncOpenAI()
    resp = await client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "Create a research plan. Output JSON with keys: "
                    "'title', 'sections' (list of section objects "
                    "with 'heading' and 'questions' list), "
                    "'estimated_sources' (int)."
                ),
            },
            {"role": "user", "content": f"Topic: {state['topic']}"},
        ],
        response_format={"type": "json_object"},
    )
    plan = json.loads(resp.choices[0].message.content)
    return {"plan": plan, "budget_used": state["budget_used"] + 0.03}


async def research_node(state: ReportState) -> dict:
    """Researcher agent: Execute research for each planned question.

    Iterates through every section and question in the plan,
    using the ResearchAgent's ReAct loop to gather findings.
    """
    agent = ResearchAgent(RESEARCH_TOOLS, max_steps=8)
    all_notes = []

    for section in state["plan"]["sections"]:
        for question in section["questions"]:
            notes = await agent.research_question(
                question,
                context=(
                    f"Report: {state['plan']['title']}, "
                    f"Section: {section['heading']}"
                ),
            )
            all_notes.extend(notes)

    return {
        "research_notes": all_notes,
        "budget_used": state["budget_used"] + len(all_notes) * 0.05,
    }


async def synthesize_node(state: ReportState) -> dict:
    """Synthesizer agent: Analyze research notes into structured findings.

    Identifies key themes, contradictions, and gaps. If there is
    previous review feedback, incorporates it into the analysis.
    """
    client = AsyncOpenAI()
    notes_text = "\n\n".join(
        f"### {n['title']}\n{n['content']}\nSources: {', '.join(n['sources'])}"
        for n in state["research_notes"]
    )

    feedback = ""
    if state.get("review_feedback"):
        feedback = (
            f"\n\nPrevious review feedback to address:\n"
            f"{state['review_feedback']}"
        )

    resp = await client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "Synthesize research notes into a structured "
                    "analysis. Identify key themes, contradictions, "
                    "and gaps. Organize by the report's planned "
                    "sections. Include all source citations."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Plan: {json.dumps(state['plan'])}\n\n"
                    f"Research Notes:\n{notes_text}{feedback}"
                ),
            },
        ],
        max_tokens=4000,
    )
    return {"analysis": resp.choices[0].message.content}


print("Agent node functions defined: plan_node, research_node, synthesize_node")

---

## 6. Report Writing Agent -- Structured Output Generation

The Writer agent transforms analysis into a polished, professional document. We use Pydantic models to enforce a strict report schema, ensuring every report has an executive summary, sections with key findings, a conclusion, and references.

In [ ]:
# ── Structured Report Schema ──


class ReportSection(BaseModel):
    """A single section of the report."""
    heading: str = Field(description="Section heading")
    content: str = Field(description="Markdown content for this section")
    key_findings: list[str] = Field(description="Bullet-point key findings")


class StructuredReport(BaseModel):
    """The complete structured report produced by the Writer agent."""
    title: str = Field(description="Report title")
    executive_summary: str = Field(description="2-paragraph executive summary")
    sections: list[ReportSection] = Field(description="Report body sections")
    conclusion: str = Field(description="Concluding remarks and recommendations")
    references: list[str] = Field(description="Numbered reference list")
    generated_at: str = Field(description="ISO timestamp of generation")


print("StructuredReport schema:")
print(json.dumps(StructuredReport.model_json_schema(), indent=2)[:500] + "...")

In [ ]:
# ── Writer Node with Structured Output ──


async def write_node(state: ReportState) -> dict:
    """Writer agent: Generate the final Markdown report.

    Uses OpenAI's structured output mode to guarantee the output
    conforms to the StructuredReport Pydantic model.
    """
    client = AsyncOpenAI()
    resp = await client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "Write a comprehensive, professional report. "
                    "Use numbered citations [1], [2] etc. Each "
                    "section should be 3-5 paragraphs with data "
                    "and evidence from the analysis."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Plan:\n{json.dumps(state['plan'])}\n\n"
                    f"Analysis:\n{state['analysis']}"
                ),
            },
        ],
        response_format=StructuredReport,
    )
    report = resp.choices[0].message.parsed
    formatter = ReportFormatter()
    return {"report": formatter.to_markdown(report)}


# ── Reviewer Node ──


async def review_node(state: ReportState) -> dict:
    """Reviewer agent: Quality check the report.

    Evaluates factual accuracy, proper citations, logical flow,
    completeness, and professional quality. Returns a pass/fail
    decision with specific feedback for revisions.
    """
    client = AsyncOpenAI()
    resp = await client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "Review this report for: factual accuracy, "
                    "proper citations, logical flow, completeness, "
                    "and professional quality. Output JSON with "
                    "'passed' (bool), 'score' (1-10), and "
                    "'feedback' (string with specific issues)."
                ),
            },
            {"role": "user", "content": state["report"]},
        ],
        response_format={"type": "json_object"},
    )
    review = json.loads(resp.choices[0].message.content)
    return {
        "review_passed": review["passed"],
        "review_feedback": review["feedback"],
        "revision_count": state["revision_count"] + 1,
    }


print("Writer and Reviewer nodes defined.")

---

## 7. Template-Based Report Formatting

The `ReportFormatter` converts the structured Pydantic model into multiple output formats: Markdown for version control and editing, and HTML for web display.

In [ ]:
class ReportFormatter:
    """Convert a StructuredReport into multiple output formats."""

    def to_markdown(self, report: StructuredReport) -> str:
        """Convert structured report to Markdown string."""
        parts = [
            f"# {report.title}\n",
            f"*Generated: {report.generated_at}*\n",
            f"## Executive Summary\n\n{report.executive_summary}\n",
            "## Table of Contents\n",
        ]

        # Build table of contents
        for i, sec in enumerate(report.sections, 1):
            parts.append(f"{i}. [{sec.heading}](#section-{i})")
        parts.append("")

        # Build sections
        for i, sec in enumerate(report.sections, 1):
            parts.append(f"## {i}. {sec.heading}\n")
            parts.append(sec.content + "\n")
            if sec.key_findings:
                parts.append("**Key Findings:**")
                for f in sec.key_findings:
                    parts.append(f"- {f}")
                parts.append("")

        # Conclusion and references
        parts.append(f"## Conclusion\n\n{report.conclusion}\n")
        parts.append("## References\n")
        for i, ref in enumerate(report.references, 1):
            parts.append(f"[{i}] {ref}")

        return "\n".join(parts)

    def to_html(self, md_content: str) -> str:
        """Convert Markdown report to styled HTML."""
        try:
            import markdown as md_lib
            html_body = md_lib.markdown(
                md_content, extensions=["tables", "toc"]
            )
        except ImportError:
            # Fallback: wrap raw markdown in <pre> tags
            html_body = f"<pre>{md_content}</pre>"

        return f"""<!DOCTYPE html>
<html><head>
<style>
  body {{ font-family: Georgia, serif; max-width: 800px;
         margin: 40px auto; padding: 0 20px; line-height: 1.8; }}
  h1 {{ color: #1a1a2e; border-bottom: 2px solid #f59e0b; }}
  h2 {{ color: #16213e; margin-top: 2em; }}
  code {{ background: #f5f5f5; padding: 2px 6px; border-radius: 3px; }}
  blockquote {{ border-left: 3px solid #f59e0b; padding-left: 1em;
                color: #555; }}
</style>
</head><body>{html_body}</body></html>"""


# ── Demo: Format a sample report ──
sample_report = StructuredReport(
    title="Sample Report: AI in Healthcare",
    executive_summary="This report examines the impact of AI on healthcare delivery.",
    sections=[
        ReportSection(
            heading="Diagnostic AI",
            content="AI-powered diagnostics have shown 94% accuracy in radiology [1].",
            key_findings=["94% accuracy in radiology", "FDA approved 500+ AI devices"],
        ),
        ReportSection(
            heading="Drug Discovery",
            content="Machine learning accelerates drug candidate identification [2].",
            key_findings=["40% reduction in discovery time", "Cost savings of $1.5B"],
        ),
    ],
    conclusion="AI will continue transforming healthcare in the coming decade.",
    references=[
        "Smith et al., 'AI in Radiology', Nature Medicine 2024",
        "Johnson, 'ML for Drug Discovery', Science 2024",
    ],
    generated_at=datetime.now().isoformat(),
)

formatter = ReportFormatter()
md_output = formatter.to_markdown(sample_report)

print("=== Markdown Output (preview) ===")
print(md_output[:600])
print("...")

---

## 8. LangGraph Multi-Agent Workflow

LangGraph orchestrates the agents as a state machine. Each node is an agent, each edge is a transition. The workflow includes:

- **Sequential edges**: planner -> researcher -> synthesizer -> writer -> reviewer
- **Conditional edge**: reviewer routes to either `END` (pass) or back to `synthesizer` (revise)
- **Interrupt**: pauses before the researcher node for human plan approval

In [ ]:
# ── Conditional Edge: Should the report be revised? ──


def should_revise(state: ReportState) -> str:
    """Determine whether to accept the report or send it back for revision.

    Rules:
    - If review passed: accept and end.
    - If max revisions (3) reached: accept as-is to prevent infinite loops.
    - Otherwise: route back to synthesizer for revision.
    """
    if state["review_passed"]:
        return "end"
    if state["revision_count"] >= 3:
        return "end"  # Max revisions reached
    return "revise"


# ── Build the LangGraph State Machine ──

graph = StateGraph(ReportState)

# Add agent nodes
graph.add_node("planner", plan_node)
graph.add_node("researcher", research_node)
graph.add_node("synthesizer", synthesize_node)
graph.add_node("writer", write_node)
graph.add_node("reviewer", review_node)

# Define the workflow edges
graph.set_entry_point("planner")
graph.add_edge("planner", "researcher")
graph.add_edge("researcher", "synthesizer")
graph.add_edge("synthesizer", "writer")
graph.add_edge("writer", "reviewer")

# Conditional edge: reviewer decides to accept or revise
graph.add_conditional_edges(
    "reviewer",
    should_revise,
    {"revise": "synthesizer", "end": END},
)

# Compile with checkpointing and HITL interrupt
checkpointer = MemorySaver()
app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["researcher"],  # HITL: pause for plan approval
)

print("LangGraph workflow compiled successfully.")
print("Nodes:", ["planner", "researcher", "synthesizer", "writer", "reviewer"])
print("Entry point: planner")
print("HITL interrupt before: researcher")
print("Conditional edge: reviewer -> {revise: synthesizer, end: END}")

### State Persistence

LangGraph's `MemorySaver` checkpointer enables state persistence. The entire state can be saved at any point and the workflow can be resumed later. This is essential for:

- **HITL checkpoint**: When pausing for human approval, state is serialized. On approval, it deserializes and continues.
- **Debugging**: Inspect state at any stage to see what each agent produced.
- **Recovery**: If the system crashes mid-pipeline, resume from the last checkpoint.

The `thread_id` in the config enables multiple concurrent report generations.

---

## 9. Guardrails -- Quality Checks & Validation

Guardrails keep the autonomous agent from going off the rails. We implement three categories:

1. **Budget guardrails**: Track and limit API spending per report
2. **Content guardrails**: Scan outputs for prohibited content (PII, overconfident claims)
3. **Citation guardrails**: Verify that all citations reference real sources

In [ ]:
# ── Budget Guardrail ──


@dataclass
class BudgetGuard:
    """Tracks and enforces API spending limits.

    Three states:
    - allowed: proceed normally
    - warning: reduce scope (approaching limit)
    - blocked: skip to synthesis (budget exhausted)
    """
    max_budget: float = 5.0   # dollars
    warn_threshold: float = 0.8  # warn at 80%

    def check(self, state: ReportState) -> dict:
        used = state["budget_used"]
        remaining = self.max_budget - used

        if used >= self.max_budget:
            return {
                "allowed": False,
                "reason": "Budget exhausted",
                "action": "skip_to_synthesis",
            }

        if used >= self.max_budget * self.warn_threshold:
            return {
                "allowed": True,
                "warning": f"Budget ${remaining:.2f} remaining",
                "action": "reduce_scope",
            }

        return {"allowed": True}


# ── Content Guardrail ──


class ContentGuard:
    """Scans agent outputs for prohibited content patterns."""

    PROHIBITED = [
        (r"(?i)this is not (legal|medical) advice", "disclaimer_needed"),
        (r"\b\d{3}-\d{2}-\d{4}\b", "ssn_detected"),
        (r"(?i)(guaranteed|certainly will|100% effective)", "overconfident_claim"),
    ]

    def scan(self, text: str) -> list[dict]:
        """Scan text for prohibited content patterns."""
        issues = []
        for pattern, issue_type in self.PROHIBITED:
            matches = re.findall(pattern, text)
            if matches:
                issues.append(
                    {
                        "type": issue_type,
                        "matches": matches[:3],
                        "severity": "high" if "ssn" in issue_type else "medium",
                    }
                )
        return issues


# ── Citation Guardrail ──


class CitationGuard:
    """Verifies that report citations reference real sources."""

    def verify(self, report: str, sources: list[dict]) -> dict:
        """Check citation validity and coverage."""
        # Find all [N] citations in the report
        cited = set(re.findall(r"\[(\d+)\]", report))
        available = set(str(i) for i in range(1, len(sources) + 1))

        invalid = cited - available
        unused = available - cited

        return {
            "valid": len(invalid) == 0,
            "invalid_citations": list(invalid),
            "unused_sources": list(unused),
            "citation_coverage": len(cited) / max(len(available), 1),
        }


# ── Demo guardrails ──
budget_guard = BudgetGuard(max_budget=5.0)
content_guard = ContentGuard()
citation_guard = CitationGuard()

# Test budget guard
test_state_ok = {"budget_used": 2.0}
test_state_warn = {"budget_used": 4.5}
test_state_over = {"budget_used": 5.5}

print("Budget Guard Tests:")
print(f"  $2.00 used: {budget_guard.check(test_state_ok)}")
print(f"  $4.50 used: {budget_guard.check(test_state_warn)}")
print(f"  $5.50 used: {budget_guard.check(test_state_over)}")

# Test content guard
test_text = "This treatment is guaranteed to work. SSN: 123-45-6789"
issues = content_guard.scan(test_text)
print(f"\nContent Guard found {len(issues)} issue(s):")
for issue in issues:
    print(f"  - {issue['type']} (severity: {issue['severity']})")

# Test citation guard
test_report = "Finding A [1] and Finding B [2] and Finding C [5]"
test_sources = [{"title": "Source 1"}, {"title": "Source 2"}, {"title": "Source 3"}]
citation_result = citation_guard.verify(test_report, test_sources)
print(f"\nCitation Guard:")
print(f"  Valid: {citation_result['valid']}")
print(f"  Invalid citations: {citation_result['invalid_citations']}")
print(f"  Unused sources: {citation_result['unused_sources']}")
print(f"  Coverage: {citation_result['citation_coverage']:.0%}")

In [ ]:
# ── Guardrailed Node Wrapper ──
# This decorator can be applied to any node function, adding budget
# pre-checks and content post-checks transparently.


def with_guardrails(node_fn, budget_guard, content_guard):
    """Wrap a node function with budget and content guardrails.

    Pre-check: Verify budget is not exhausted.
    Post-check: Scan output for prohibited content.
    """
    async def wrapped(state):
        # Pre-check: budget
        budget = budget_guard.check(state)
        if not budget["allowed"]:
            return {"errors": state["errors"] + [budget["reason"]]}

        # Execute the node
        result = await node_fn(state)

        # Post-check: content
        for key in ["report", "analysis"]:
            if key in result:
                issues = content_guard.scan(result[key])
                if any(i["severity"] == "high" for i in issues):
                    result["errors"] = state.get("errors", []) + [
                        f"Content issue: {i['type']}" for i in issues
                    ]
        return result

    return wrapped


# Example: wrapping the synthesize node with guardrails
guarded_synthesize = with_guardrails(synthesize_node, budget_guard, content_guard)

print("Guardrailed wrapper defined.")
print("Usage: guarded_node = with_guardrails(node_fn, budget_guard, content_guard)")

---

## 10. Human Review Integration

Human-in-the-loop (HITL) is the ultimate guardrail. In our system, the human reviews the research plan before the agent spends time and money executing it. The `interrupt_before=["researcher"]` in the graph compilation pauses execution at the right moment.

The controlling application manages the interruption:

In [ ]:
async def run_report_agent(topic: str) -> str:
    """Run the full report agent pipeline with human-in-the-loop.

    Phase 1: Planner runs, then execution pauses for human approval.
    Phase 2: On approval, the pipeline resumes through research,
             synthesis, writing, and review.

    Args:
        topic: The report topic to research.

    Returns:
        The final Markdown report.
    """
    config = {"configurable": {"thread_id": "report-1"}}
    initial = {
        "topic": topic,
        "budget_used": 0.0,
        "revision_count": 0,
        "errors": [],
        "research_notes": [],
        "plan_approved": False,
    }

    # Phase 1: Run until HITL interrupt (after planner, before researcher)
    state = await app.ainvoke(initial, config)

    # Display plan to user for review
    print("=== Research Plan ===")
    print(json.dumps(state["plan"], indent=2))

    # Get human approval
    # In a notebook, we simulate approval. In production, this would be
    # a UI prompt, WebSocket message, or API callback.
    approval = input("Approve plan? (yes/no/edit): ")

    if approval.lower() == "no":
        return "Plan rejected by user."

    if approval.lower() == "edit":
        edits = input("Enter modifications: ")
        state["topic"] = state["topic"] + f"\n\nUser modifications: {edits}"

    # Update state with approval
    await app.aupdate_state(config, {"plan_approved": True})

    # Phase 2: Resume execution (research -> synthesis -> write -> review)
    final_state = await app.ainvoke(None, config)
    return final_state["report"]


print("run_report_agent() defined.")
print("This function orchestrates the full pipeline with human approval.")
print("")
print("Usage:")
print('  report = await run_report_agent("AI in Healthcare 2025")')

---

## 11. Evaluation Framework

Before deploying the report agent, we need to assess quality at multiple levels. The evaluator combines:

- **Structural checks**: Does the report have all required sections?
- **Citation metrics**: What percentage of claims have citations?
- **LLM-as-judge**: A separate LLM scores clarity, depth, accuracy, and actionability.
- **Efficiency metrics**: Cost per word, revision count.

In [ ]:
class ReportEvaluator:
    """Multi-dimensional evaluation of generated reports."""

    def __init__(self):
        self.client = AsyncOpenAI()

    async def evaluate(self, state: ReportState) -> dict:
        """Evaluate a report across multiple quality dimensions.

        Returns a dict with structural, citation, LLM-judge, and
        efficiency scores.
        """
        scores = {}
        report = state["report"]

        # 1. Structural completeness
        scores["has_executive_summary"] = "executive summary" in report.lower()
        scores["has_conclusion"] = "conclusion" in report.lower()
        scores["has_references"] = "references" in report.lower()
        scores["word_count"] = len(report.split())

        # 2. Citation coverage
        citations = re.findall(r"\[\d+\]", report)
        paragraphs = [p for p in report.split("\n\n") if len(p) > 100]
        cited_paras = [p for p in paragraphs if re.search(r"\[\d+\]", p)]
        scores["citation_density"] = len(citations) / max(len(paragraphs), 1)
        scores["cited_para_ratio"] = len(cited_paras) / max(len(paragraphs), 1)

        # 3. LLM-as-judge (multi-dimension scoring)
        resp = await self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Score this report 1-10 on each dimension. "
                        "Output JSON: {clarity, depth, accuracy, "
                        "actionability, citations, overall, feedback}"
                    ),
                },
                {
                    "role": "user",
                    "content": f"Topic: {state['topic']}\n\n{report}",
                },
            ],
            response_format={"type": "json_object"},
        )
        llm_scores = json.loads(resp.choices[0].message.content)
        scores["llm_eval"] = llm_scores

        # 4. Budget efficiency
        scores["cost"] = state["budget_used"]
        scores["cost_per_word"] = state["budget_used"] / max(
            scores["word_count"], 1
        )
        scores["revisions"] = state["revision_count"]

        return scores


print("ReportEvaluator defined.")
print("")
print("Evaluation dimensions and targets:")
print("  Clarity:           8+ / 10")
print("  Depth:             7+ / 10")
print("  Accuracy:          9+ / 10")
print("  Actionability:     7+ / 10")
print("  Citation Coverage: > 80%")
print("  Cost Efficiency:   < $3 / report")

---

## 12. End-to-End Demo with Sample Data

Since the full pipeline requires live API calls, we demonstrate the workflow with **simulated state** to show how data flows through each stage. This lets you test the architecture locally without API costs.

In [ ]:
# ── Simulated End-to-End Pipeline ──
# This demonstrates the data flow through each stage without live API calls.

# Step 1: Simulated Planner output
sample_plan = {
    "title": "The Impact of AI on Healthcare in 2025",
    "sections": [
        {
            "heading": "AI-Powered Diagnostics",
            "questions": [
                "What AI diagnostic tools are FDA-approved in 2025?",
                "How accurate are AI diagnostics compared to physicians?",
            ],
        },
        {
            "heading": "Drug Discovery and Development",
            "questions": [
                "How is ML accelerating drug candidate identification?",
                "What are the cost savings from AI in pharma R&D?",
            ],
        },
        {
            "heading": "Patient Care and Operations",
            "questions": [
                "How are hospitals using AI for operational efficiency?",
                "What role does AI play in personalized treatment plans?",
            ],
        },
    ],
    "estimated_sources": 15,
}

print("Step 1 - Planner Output:")
print(f"  Title: {sample_plan['title']}")
print(f"  Sections: {len(sample_plan['sections'])}")
for sec in sample_plan["sections"]:
    print(f"    - {sec['heading']} ({len(sec['questions'])} questions)")
print(f"  Estimated sources: {sample_plan['estimated_sources']}")


# Step 2: Simulated Research notes
sample_notes = [
    {
        "title": "FDA-Approved AI Diagnostics",
        "content": (
            "As of 2025, over 500 AI/ML-enabled medical devices have received "
            "FDA clearance. Radiology leads with 75% of approvals, followed by "
            "cardiology (12%) and ophthalmology (8%). Notable devices include "
            "IDx-DR for diabetic retinopathy screening."
        ),
        "sources": ["FDA AI/ML Device Database 2025", "Nature Medicine Review 2025"],
    },
    {
        "title": "AI Diagnostic Accuracy",
        "content": (
            "Meta-analysis of 82 studies shows AI diagnostic systems achieve "
            "87.0% sensitivity and 92.5% specificity on average, comparable to "
            "senior specialists. However, performance varies significantly by "
            "clinical domain and data quality."
        ),
        "sources": ["Lancet Digital Health 2024", "JAMA AI Review 2025"],
    },
    {
        "title": "ML in Drug Discovery",
        "content": (
            "Machine learning has reduced average drug discovery timelines from "
            "4.5 years to 2.7 years for lead identification. Companies like "
            "Insilico Medicine and Recursion Pharmaceuticals have advanced "
            "AI-discovered candidates to Phase II clinical trials."
        ),
        "sources": ["Drug Discovery Today 2025", "McKinsey Pharma Report 2025"],
    },
    {
        "title": "Hospital AI Operations",
        "content": (
            "Leading hospital systems report 15-30% improvement in bed utilization, "
            "20% reduction in patient wait times, and $2-5M annual savings from "
            "AI-powered scheduling and resource optimization."
        ),
        "sources": ["Health Affairs 2025", "HIMSS Analytics 2025"],
    },
]

print(f"\nStep 2 - Researcher Output: {len(sample_notes)} research notes")
for note in sample_notes:
    print(f"  - {note['title']} ({len(note['sources'])} sources)")

In [ ]:
# Step 3: Simulated synthesis
sample_analysis = """
## Analysis: AI in Healthcare 2025

### Theme 1: Diagnostic AI is Mature but Uneven
AI diagnostics have reached production maturity with 500+ FDA-cleared devices [1].
Accuracy is comparable to senior specialists (87% sensitivity, 92.5% specificity) [2],
but performance varies by domain and data quality.

### Theme 2: Drug Discovery Acceleration
ML has cut lead identification timelines by 40% (4.5 to 2.7 years) [3].
Multiple AI-discovered candidates are in Phase II trials, suggesting
the approach is commercially viable.

### Theme 3: Operational Efficiency Gains
Hospital AI systems deliver measurable ROI: 15-30% bed utilization improvement,
20% wait time reduction, and $2-5M annual savings [4].

### Gap: Equity and Access
Research notes lack coverage of AI's impact on healthcare equity and
access in underserved communities. This should be flagged.
""".strip()

# Step 4: Generate report using the formatter
sample_structured = StructuredReport(
    title="The Impact of AI on Healthcare in 2025",
    executive_summary=(
        "Artificial intelligence is reshaping healthcare across diagnostics, "
        "drug discovery, and hospital operations. With over 500 FDA-cleared "
        "AI devices and documented cost savings of $2-5M annually for leading "
        "hospital systems, the technology has moved beyond pilot programs into "
        "production deployment.\n\n"
        "However, significant challenges remain in ensuring equitable access, "
        "validating performance across diverse populations, and integrating AI "
        "tools into existing clinical workflows without disrupting care quality."
    ),
    sections=[
        ReportSection(
            heading="AI-Powered Diagnostics",
            content=(
                "The FDA has cleared over 500 AI/ML-enabled medical devices as of "
                "2025, with radiology accounting for 75% of approvals [1]. These tools "
                "have demonstrated diagnostic accuracy comparable to senior specialists, "
                "with meta-analyses showing 87.0% sensitivity and 92.5% specificity "
                "across 82 studies [2].\n\n"
                "Notable deployments include IDx-DR for autonomous diabetic retinopathy "
                "screening, which operates without physician oversight in primary care "
                "settings. However, performance varies significantly by clinical domain "
                "and the quality of training data available."
            ),
            key_findings=[
                "500+ FDA-cleared AI medical devices (75% in radiology)",
                "87% sensitivity, 92.5% specificity comparable to specialists",
                "Performance varies by domain and data quality",
            ],
        ),
        ReportSection(
            heading="Drug Discovery and Development",
            content=(
                "Machine learning has reduced average drug discovery timelines from "
                "4.5 years to 2.7 years for lead identification, a 40% reduction [3]. "
                "Companies like Insilico Medicine and Recursion Pharmaceuticals have "
                "advanced AI-discovered drug candidates to Phase II clinical trials.\n\n"
                "The economic impact is substantial: industry estimates suggest AI-driven "
                "approaches could save $1.5 billion per approved drug by reducing failure "
                "rates in early-stage screening and optimizing clinical trial design."
            ),
            key_findings=[
                "40% reduction in lead identification timelines",
                "AI-discovered candidates reaching Phase II trials",
                "Potential $1.5B savings per approved drug",
            ],
        ),
        ReportSection(
            heading="Patient Care and Operations",
            content=(
                "Leading hospital systems report significant operational improvements "
                "from AI deployment: 15-30% improvement in bed utilization, 20% "
                "reduction in patient wait times, and $2-5M in annual cost savings [4].\n\n"
                "AI-powered scheduling and resource optimization tools are among the "
                "most widely adopted applications, providing measurable ROI within "
                "12-18 months of deployment."
            ),
            key_findings=[
                "15-30% bed utilization improvement",
                "20% reduction in patient wait times",
                "$2-5M annual savings for leading systems",
            ],
        ),
    ],
    conclusion=(
        "AI in healthcare has transitioned from experimental to operational, with "
        "measurable impact across diagnostics, drug discovery, and hospital management. "
        "Organizations should prioritize validated AI tools with strong clinical evidence, "
        "invest in data infrastructure, and address equity gaps to ensure broad benefit."
    ),
    references=[
        "FDA AI/ML Device Database, 2025. https://fda.gov/ai-devices",
        "Chen et al., 'AI Diagnostic Accuracy Meta-Analysis', Lancet Digital Health, 2024",
        "Zhavoronkov et al., 'ML in Drug Discovery', Drug Discovery Today, 2025",
        "HIMSS Analytics, 'AI ROI in Hospital Operations', Health Affairs, 2025",
    ],
    generated_at=datetime.now().isoformat(),
)

# Format the report
formatter = ReportFormatter()
final_md = formatter.to_markdown(sample_structured)

print("Step 3 - Synthesizer completed analysis")
print(f"Step 4 - Writer produced report ({len(final_md)} chars)")
print("\n" + "=" * 60)
print("FINAL REPORT (Markdown Preview)")
print("=" * 60)
print(final_md)

In [ ]:
# Step 5: Evaluate the report (simulated state)
eval_state = {
    "topic": "AI in Healthcare 2025",
    "report": final_md,
    "budget_used": 1.85,
    "revision_count": 1,
}

# Run structural and citation checks (no API call needed)
eval_scores = {}
eval_scores["has_executive_summary"] = "executive summary" in final_md.lower()
eval_scores["has_conclusion"] = "conclusion" in final_md.lower()
eval_scores["has_references"] = "references" in final_md.lower()
eval_scores["word_count"] = len(final_md.split())

citations = re.findall(r"\[\d+\]", final_md)
paragraphs = [p for p in final_md.split("\n\n") if len(p) > 100]
cited_paras = [p for p in paragraphs if re.search(r"\[\d+\]", p)]
eval_scores["citation_count"] = len(citations)
eval_scores["citation_density"] = len(citations) / max(len(paragraphs), 1)
eval_scores["cited_para_ratio"] = len(cited_paras) / max(len(paragraphs), 1)
eval_scores["cost"] = eval_state["budget_used"]
eval_scores["cost_per_word"] = eval_state["budget_used"] / max(
    eval_scores["word_count"], 1
)

# Verify citations against sources
citation_check = citation_guard.verify(final_md, sample_structured.references)

print("=== Report Evaluation ===")
print(f"\nStructural Completeness:")
print(f"  Executive Summary: {eval_scores['has_executive_summary']}")
print(f"  Conclusion:        {eval_scores['has_conclusion']}")
print(f"  References:        {eval_scores['has_references']}")
print(f"  Word Count:        {eval_scores['word_count']}")

print(f"\nCitation Quality:")
print(f"  Total Citations:   {eval_scores['citation_count']}")
print(f"  Citation Density:  {eval_scores['citation_density']:.2f} per paragraph")
print(f"  Cited Paragraphs:  {eval_scores['cited_para_ratio']:.0%}")
print(f"  All Citations Valid: {citation_check['valid']}")
print(f"  Coverage:          {citation_check['citation_coverage']:.0%}")

print(f"\nEfficiency:")
print(f"  Total Cost:        ${eval_scores['cost']:.2f}")
print(f"  Cost per Word:     ${eval_scores['cost_per_word']:.4f}")
print(f"  Revisions:         {eval_state['revision_count']}")

---

## 13. Export to Different Formats

The report can be exported to multiple formats for different use cases:
- **Markdown**: For version control, editing, and rendering in documentation platforms
- **HTML**: For web display with professional styling

In [ ]:
# ── Export to Markdown ──
md_path = Path("sample_report.md")
md_path.write_text(final_md)
print(f"Markdown report saved to: {md_path.absolute()}")
print(f"  Size: {md_path.stat().st_size:,} bytes")


# ── Export to HTML ──
html_content = formatter.to_html(final_md)
html_path = Path("sample_report.html")
html_path.write_text(html_content)
print(f"\nHTML report saved to: {html_path.absolute()}")
print(f"  Size: {html_path.stat().st_size:,} bytes")


# ── Preview HTML structure ──
print("\n=== HTML Preview (first 500 chars) ===")
print(html_content[:500])
print("...")

In [ ]:
# ── Programmatic Export Function ──


def export_report(
    report: StructuredReport,
    output_dir: str = ".",
    formats: list[str] = None,
) -> dict[str, str]:
    """Export a structured report to multiple formats.

    Args:
        report: The StructuredReport to export.
        output_dir: Directory to save files.
        formats: List of formats to export ('markdown', 'html').
                 Defaults to both.

    Returns:
        Dict mapping format names to file paths.
    """
    if formats is None:
        formats = ["markdown", "html"]

    formatter = ReportFormatter()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Create a safe filename from the title
    safe_title = re.sub(r"[^\w\s-]", "", report.title).strip().replace(" ", "_")
    safe_title = safe_title[:50]  # Limit length

    outputs = {}

    if "markdown" in formats:
        md_content = formatter.to_markdown(report)
        md_file = output_dir / f"{safe_title}.md"
        md_file.write_text(md_content)
        outputs["markdown"] = str(md_file)

    if "html" in formats:
        md_content = formatter.to_markdown(report)
        html_content = formatter.to_html(md_content)
        html_file = output_dir / f"{safe_title}.html"
        html_file.write_text(html_content)
        outputs["html"] = str(html_file)

    return outputs


# Demo export
exported = export_report(sample_structured, output_dir="./reports")
print("Exported report files:")
for fmt, path in exported.items():
    print(f"  {fmt}: {path}")

---

## 14. Running the Full Pipeline (Live)

To run the complete pipeline with live API calls, uncomment and execute the cell below. This requires:
- A valid `OPENAI_API_KEY` environment variable
- (Optional) A valid `TAVILY_API_KEY` for web search
- (Optional) A running RAG service on `localhost:8000` for document retrieval

**Estimated cost**: $1-3 per report depending on topic complexity.

In [ ]:
# ── Uncomment to run the live pipeline ──

# import asyncio
#
# async def main():
#     report = await run_report_agent("The Impact of AI on Healthcare in 2025")
#     print(report)
#
#     # Export the result
#     # Note: For live runs, you would parse the markdown back into
#     # a StructuredReport or save the raw markdown directly.
#     Path("live_report.md").write_text(report)
#     print("\nReport saved to live_report.md")
#
# asyncio.run(main())

print("Live pipeline cell ready (uncomment to run).")
print("")
print("To run in a Jupyter notebook, use:")
print('  report = await run_report_agent("Your Topic Here")')

---

## Summary

This notebook demonstrated how to build an **Autonomous Report Agent** that integrates nearly every concept from the CareerAlign GenAI Course:

| Component | Concept | Module |
|-----------|---------|--------|
| Agent brains | LLM API calls (planning, synthesis, writing, review) | 03 |
| System prompts | Prompt engineering for specialized agent roles | 06 |
| Research tools | Web search, RAG retrieval, data APIs | 07, 09 |
| Orchestration | LangGraph state machine, conditional edges, interrupts | 09 |
| Safety | Budget, content, and citation guardrails | 11 |
| Quality | Multi-dimensional evaluation, LLM-as-judge | 10 |
| Output | Structured reports in Markdown and HTML | 04 |

### Key Takeaways

1. **Supervisor pattern** provides predictability and debuggability over autonomous swarms.
2. **State machines** (LangGraph) make complex multi-agent workflows manageable and observable.
3. **Guardrails are not optional** -- budget limits, content filters, and human checkpoints are essential for production agents.
4. **Structured output** (Pydantic models) eliminates fragile JSON parsing and ensures consistency.
5. **Evaluation must be multi-dimensional** -- no single metric captures report quality.

### Next Steps

- Add PDF export using WeasyPrint
- Implement a FastAPI + WebSocket service for real-time progress
- Deploy with Docker alongside the Document Portal (Capstone I)
- Add more research tools (academic paper search, financial data APIs)
- Fine-tune the writer prompt for domain-specific report styles